# Skyline Matching Evaluation (Colab)

## Before running

Upload these to Google Drive under `SkylineGeolocation/`:
- `notebooks/02_SkylineDatabase/output/skyline_db.parquet` (911 MB)
- `data/synthetic_dataset/` (267 MB, whole folder)

Everything else (code, ground_truth.json) comes from the repo.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!apt-get install -qq libgl1-mesa-glx libglib2.0-0  # opencv deps
!pip install -q fastdtw pyarrow geopy pyprojroot segmentation-models-pytorch timm albumentations

In [ ]:
import os, sys, shutil
from pathlib import Path

# Symlink data from Drive to where the code expects it
DRIVE_BASE = Path("/content/drive/MyDrive/SkylineGeolocation")

for src, dst in [
    (DRIVE_BASE / "notebooks", "notebooks"),
    (DRIVE_BASE / "data", "data"),
]:
    if src.exists():
        dest_path = Path(dst)
        if not dest_path.exists():
            shutil.copytree(src, dest_path, symlinks=True)
            print(f"Copied {src} -> {dst}")
        else:
            print(f"{dst} already exists")

# Clone repo if needed (for src/ and notebooks/)
if not Path("src").exists():
    !git clone https://github.com/ppras/SkylineGeolocation.git /tmp/repo
    !cp -r /tmp/repo/src /tmp/repo/notebooks /tmp/repo/tests /tmp/repo/scripts .

sys.path.insert(0, ".")

In [ ]:
# Verify data
import os
db = "notebooks/02_SkylineDatabase/output/skyline_db.parquet"
gt = "data/synthetic_dataset/ground_truth.json"
masks = "data/synthetic_dataset/predicted_masks"
print(f"DB: {os.path.getsize(db)/1e6:.0f} MB" if os.path.exists(db) else "MISSING: " + db)
print(f"GT: {os.path.getsize(gt)/1e6:.1f} MB" if os.path.exists(gt) else "MISSING: " + gt)
if os.path.exists(masks):
    print(f"Masks: {len(os.listdir(masks))} files")
else:
    print("MISSING: " + masks)

In [ ]:
import sys, gc, json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
sys.path.insert(0, ".")

# Load GT
with open("data/synthetic_dataset/ground_truth.json") as f:
    gt_data = json.load(f)
sample_ids = list(gt_data.keys())
print(f"Samples: {len(sample_ids)}", flush=True)

# Load DB metadata
meta = pd.read_parquet("notebooks/02_SkylineDatabase/output/skyline_db.parquet",
                       columns=["lon", "lat", "elevation_m"])
lon_arr, lat_arr = meta["lon"].to_numpy(), meta["lat"].to_numpy()
del meta; gc.collect()
print(f"DB: {len(lon_arr)} viewpoints", flush=True)

# Get bin size
pf = pq.ParquetFile("notebooks/02_SkylineDatabase/output/skyline_db.parquet")
first = next(pf.iter_batches(batch_size=1, columns=["raw_horizon_deg"]))
bin_deg = 360.0 / len(first.to_pandas()["raw_horizon_deg"].iloc[0])
print(f"bin_deg={bin_deg}", flush=True)

In [ ]:
from src.evaluation import run_evaluation

df, summary = run_evaluation(
    ground_truth_path="data/synthetic_dataset/ground_truth.json",
    db_path="notebooks/02_SkylineDatabase/output/skyline_db.parquet",
    masks_dir="data/synthetic_dataset/predicted_masks",
    use_altimeter=True,
    use_compass=True,
    limit=0,
    sample_batch_size=8,
    top_k=30,
    dtw_window=15,
    correct_dist_m=500.0,
    chunk_rows=4000,
    spatial_stride=5,
)

print(json.dumps(summary, indent=2))
if len(df) > 0:
    df.to_csv("eval_results.csv", index=False)
    print(f"Saved eval_results.csv ({len(df)} rows)")

In [ ]:
import matplotlib.pyplot as plt

if len(df) > 0:
    errors = df["error_m"]
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.hist(errors, bins=50, color="steelblue", edgecolor="white")
    plt.axvline(500, color="red", ls="--", label="500m")
    plt.xlabel("Error (m)")
    plt.ylabel("Count")
    plt.title(f"Top-1 Errors (n={len(errors)})")
    plt.legend()

    plt.subplot(1, 2, 2)
    for dist, label in [(100, "100m"), (500, "500m"), (1000, "1km"), (5000, "5km")]:
        acc = (errors <= dist).mean() * 100
        plt.bar(label, acc, color="seagreen")
    plt.ylabel("Top-1 Accuracy (%)")
    plt.title("Accuracy at Distance Thresholds")
    plt.tight_layout()
    plt.savefig("eval_results.png", dpi=150)
    plt.show()
    print(f"Median error: {errors.median():.0f}m")
    print(f"Top-1@500m:  {(errors <= 500).mean()*100:.1f}%")

## Results

`eval_results.csv` and `eval_results.png` were saved. Download them from Colab's file browser.